## IBOV

In [0]:
import time
import logging
from datetime import datetime
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType
from config import ROUTES, PipelineConfig

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# confing
NOME_TABELA  = f"bronze_ibov_index" 
BRONZE_PATH = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
URL          = r"https://query1.finance.yahoo.com/v8/finance/chart/%5EBVSP?range=10y&interval=1d"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

In [0]:
SCHEMA_IBOV = StructType([
    StructField("timestamp", StringType(), True),
    StructField("close", StringType(), True),
])

# 1. Extração

log.info(f"Iniciando ingestão | série=BVSP | nome=IBOVESPA | frequencia=diario data_processamento={DATA_PROC}")

data = PipelineConfig.retorno_json_url(url=URL)

# 2. Criação do DF com esquema definido 
timestamps = data["chart"]["result"][0]["timestamp"]
close = data["chart"]["result"][0]["indicators"]["quote"][0]["close"]

# criar lista de registros
records = list(zip(timestamps, close))
log.info(f"Registros recebidos da API: {len(records)}")

df = spark.createDataFrame(records, schema=SCHEMA_IBOV)

# Metadados de rastreabilidade 
df = (df
      .withColumn("_source_url", f.lit(URL))
      .withColumn("_ingest_timestamp", f.current_timestamp())
      .withColumn("data_processamento", f.lit(DATA_PROC))
)



In [0]:
try:
    # 4. Escrita na bronze
    n = df.count()

    log.info(f"Escrevendo {n} linhas em Bronze ")

    (df.coalesce(1)
    .write
    .mode("overwrite")
    .option("replaceWhere", f"data_processamento = {DATA_PROC}")
    .option("mergeSchema",  "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .partitionBy("data_processamento")
    .format("delta")
    .saveAsTable(BRONZE_PATH)
    )

    PipelineConfig.registrar_auditoria(
        spark, ROUTES.AUDIT_PATH, "bronze_fechamento_ibovespa",
        BRONZE_PATH, n, "SUCESSO", DATA_PROC
    )

except Exception as e:
    PipelineConfig.registrar_auditoria(
        spark, ROUTES.AUDIT_PATH, "bronze_fechamento_ibovespa",
        BRONZE_PATH, 0, "FALHA", DATA_PROC, str(e)
    )
    raise

# 4. metricas
log.info(f"Ingestão concluida | série=BVSP | tabela={NOME_TABELA} | linhas={n_registros} | partição={DATA_PROC}")